# M5 Forecasting — ETL Pipeline: **Extract Phase**

**Project:** Retail-Demand-Forecasting
**Stage:** `01_extract` — data acquisition & exploratory profiling only
**Author:** Data Engineering / Analytics Team

---

## Purpose of this notebook

This notebook implements the **Extract** phase of an Extract–Transform–Load (ETL) pipeline for the
M5 Forecasting dataset. Its only job is to:

1. Locate the raw source files reliably, regardless of where the notebook is launched from.
2. Load the three raw M5 tables (`calendar.csv`, `sell_prices.csv`, `sales_train_validation.csv`).
3. **Profile** each dataset thoroughly (structure, types, memory, quality) so that downstream
   Transform-phase decisions are made with full knowledge of the data.
4. Document the schema, relationships, keys, and known data-quality issues.

> ⚠️ **Scope boundary — read this first**
>
> This notebook is **exploratory / diagnostic only**.
> - No missing values are imputed.
> - No columns are renamed, cast, dropped, or reshaped.
> - No merges/joins are materialized (relationships are only *inspected*).
> - No files are written back to disk.
>
> Anything that changes the data belongs in the **Transform** phase notebook
> (e.g. `notebooks/02_transform.ipynb`), not here. Keeping Extract "read-only" means we can
> re-run it safely at any time as a data-quality audit, and it gives the Transform phase a
> single, well-documented source of truth to build on.


## 1. Imports

Only lightweight, standard-library-adjacent tools are needed for extraction and profiling:
`pathlib` for filesystem-safe path resolution, `pandas`/`numpy` for tabular loading and
inspection, and `warnings` to keep the profiling output readable.


In [9]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

print(f"pandas version : {pd.__version__}")
print(f"numpy version  : {np.__version__}")
print(f"python version : {sys.version.split()[0]}")


pandas version : 3.0.5
numpy version  : 2.5.1
python version : 3.12.7


**What was done:** Imported `pathlib.Path` (for OS-independent path handling),
`pandas`/`numpy` (for data loading and profiling), and suppressed noisy runtime warnings for
cleaner output.

**Why it matters:** Using `pathlib` instead of hardcoded strings (`"C:\\Users\\..."` or
`"/home/user/..."`) is what makes this notebook portable across machines, OSes, and CI
runners — a hard requirement for anything "production-oriented."

**ETL decision influenced:** Establishes the tooling baseline for the whole Extract phase.
No heavy libraries (e.g. Dask, Spark) are introduced yet — the dataset size will be measured
later in this notebook, and *that* measurement (not assumption) will drive whether the
Transform phase needs out-of-core / chunked processing.


## 2. Locate the Project Root

The notebook lives in `Retail-Demand-Forecasting/notebooks/`. We resolve the project root
**relative to this notebook's location** rather than relying on the current working directory
(which changes depending on how Jupyter/VS Code/CI was launched). We walk upward from the
notebook's folder until we find a directory that actually contains `data/raw/`, which acts as
a lightweight "marker" for the project root.


In [10]:
def find_project_root(marker: str = "data/raw", start: Path | None = None) -> Path:
    '''Walk upward from `start` until a directory containing `marker` is found.

    This avoids hardcoding absolute paths and avoids relying on the notebook's current
    working directory, which is unreliable across editors/CI runners.
    '''
    # In a standard Jupyter session, `__vsc_ipynb_file__` / `__file__` are unavailable,
    # so we fall back to the current working directory of the running kernel, which for a
    # notebook opened at notebooks/extract.ipynb is normally the `notebooks/` folder itself.
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / marker).exists():
            return candidate

    raise FileNotFoundError(
        f"Could not locate project root containing '{marker}'. "
        f"Started searching from: {current}"
    )


PROJECT_ROOT = find_project_root()
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "m5"
SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw data dir : {DATA_RAW_DIR}")
print(f"src dir      : {SRC_DIR}")
print(f"notebooks dir: {NOTEBOOKS_DIR}")

assert DATA_RAW_DIR.exists(), f"Raw data directory not found at {DATA_RAW_DIR}"


Project root : D:\Mlprojects\Forecasting\Retail-Demand-Forecasting
Raw data dir : D:\Mlprojects\Forecasting\Retail-Demand-Forecasting\data\raw\m5
src dir      : D:\Mlprojects\Forecasting\Retail-Demand-Forecasting\src
notebooks dir: D:\Mlprojects\Forecasting\Retail-Demand-Forecasting\notebooks


**What was done:** Implemented a marker-based project-root finder that walks up the
directory tree from the current working directory until it finds a folder containing
`data/raw/`, then derived `DATA_RAW_DIR`, `SRC_DIR`, and `NOTEBOOKS_DIR` from it.

**Why it matters:** Absolute paths (`/Users/name/...`) break the moment the repo is cloned by
someone else, moved, or run inside Docker/CI. A marker-based search means this notebook works
identically on a laptop, a teammate's machine, or a scheduled pipeline runner — with **zero**
path edits.

**ETL decision influenced:** All file I/O in this notebook — and in every downstream
Transform/Load notebook or `src/` module — should resolve paths through this same pattern
(or a shared `src/paths.py` helper built from it), rather than each notebook reinventing path
logic. This is worth promoting into `src/` once the ETL pipeline is code-ified.


## 3. Raw File Inventory

Before loading anything into memory, we confirm which files actually exist in `data/raw/`
and record their on-disk size. This catches missing/misnamed files early, with a clear error
message, instead of a confusing `pandas` traceback later.


In [11]:
EXPECTED_FILES = {
    "calendar": "calendar.csv",
    "sell_prices": "sell_prices.csv",
    "sales_train_validation": "sales_train_validation.csv",
}

inventory_rows = []
for logical_name, filename in EXPECTED_FILES.items():
    fpath = DATA_RAW_DIR / filename
    exists = fpath.exists()
    size_mb = fpath.stat().st_size / (1024 ** 2) if exists else np.nan
    inventory_rows.append(
        {"dataset": logical_name, "filename": filename, "exists": exists, "size_MB": size_mb}
    )

file_inventory = pd.DataFrame(inventory_rows)
display(file_inventory)

missing = file_inventory.loc[~file_inventory["exists"], "filename"].tolist()
if missing:
    raise FileNotFoundError(f"Missing expected raw file(s) in {DATA_RAW_DIR}: {missing}")
print("All expected raw files are present.")


,dataset,filename,exists,size_MB
0,calendar,calendar.csv,True,0.10
1,sell_prices,sell_prices.csv,True,193.97
2,sales_train_validation,sales_train_validation.csv,True,114.45


All expected raw files are present.


**What was done:** Verified, file-by-file, that `calendar.csv`, `sell_prices.csv`, and
`sales_train_validation.csv` exist under `data/raw/`, and recorded their raw file sizes on
disk before touching `pandas`.

**Why it matters:** A "fail fast, fail clearly" check here means a missing/misnamed file is
reported as a single readable error at the top of the pipeline, instead of an obscure
`FileNotFoundError` buried inside a `read_csv` call several cells down.

**ETL decision influenced:** This inventory step is a natural candidate for a small
`src/extract.py` utility (`validate_raw_files()`) that a future scheduled ETL job can call
before attempting extraction, so the pipeline never proceeds on incomplete input.


## 4. Load the Raw Datasets

We load all three files as-is with `pandas.read_csv`, applying **no dtype coercion, no
parsing, no renaming**. The goal of the Extract phase is to bring the data into memory
faithfully so it can be profiled; any type optimization (e.g. downcasting `d_1..d_1913` to
`int16`, converting `date` to `datetime64`) is a **Transform**-phase decision that will be
made later, informed by the profiling below.


In [12]:
calendar_df = pd.read_csv(DATA_RAW_DIR / "calendar.csv")
sell_prices_df = pd.read_csv(DATA_RAW_DIR / "sell_prices.csv")
sales_df = pd.read_csv(DATA_RAW_DIR / "sales_train_validation.csv")

raw_datasets = {
    "calendar": calendar_df,
    "sell_prices": sell_prices_df,
    "sales_train_validation": sales_df,
}

print("Loaded datasets:")
for name, df in raw_datasets.items():
    print(f"  - {name:<24} shape={df.shape}")


Loaded datasets:
  - calendar                 shape=(1969, 14)
  - sell_prices              shape=(6841121, 4)
  - sales_train_validation   shape=(30490, 1919)


**What was done:** Loaded all three CSVs into `pandas` DataFrames with default settings
(no explicit `dtype=`, no `parse_dates=`, no column filtering) and collected them into a
`raw_datasets` dict for convenient iteration in later cells.

**Why it matters:** Loading with defaults lets `pandas` show us its own type inference, which
is itself useful diagnostic information — e.g. if `pandas` infers `object` for a column we
expect to be numeric, that is a data-quality signal (mixed types, stray strings, etc.) worth
catching now rather than after a silent, incorrect Transform-phase cast.

**ETL decision influenced:** Confirms all three files load successfully and establishes the
canonical in-memory objects (`calendar_df`, `sell_prices_df`, `sales_df`) referenced throughout
the rest of this notebook and by the Transform phase.


## 5. Profiling Utilities

To apply an **identical** profiling routine to all three datasets (head, shape, columns,
`info()`, `describe()`, memory usage, missing values, duplicates), we define one reusable
function instead of repeating the same eight cells three times. This keeps the notebook
consistent and makes it trivial to profile a fourth dataset later (e.g. `sales_train_evaluation.csv`)
by calling the same function.


In [13]:
def profile_dataset(df: pd.DataFrame, name: str, n_head: int = 5) -> None:
    '''Print a standard exploratory profile for a raw DataFrame.'''
    print("=" * 90)
    print(f"DATASET: {name}")
    print("=" * 90)

    print(f"\n--- shape ---\n{df.shape[0]:,} rows x {df.shape[1]:,} columns")

    print(f"\n--- columns ---\n{list(df.columns)}")

    print(f"\n--- head({n_head}) ---")
    display(df.head(n_head))

    print("\n--- info() ---")
    df.info(memory_usage="deep")

    print("\n--- describe(include='all').T ---")
    display(df.describe(include="all").T)

    mem_deep_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"\n--- memory usage (deep) ---\n{mem_deep_mb:,.2f} MB total")
    display((df.memory_usage(deep=True) / (1024 ** 2)).rename("MB").to_frame())

    print("\n--- missing values ---")
    missing_counts = df.isna().sum()
    missing_pct = (missing_counts / len(df) * 100).round(2)
    missing_report = (
        pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_pct})
        .query("missing_count > 0")
        .sort_values("missing_count", ascending=False)
    )
    if missing_report.empty:
        print("No missing values detected in any column.")
    else:
        display(missing_report)

    print("\n--- duplicate rows ---")
    n_dupes = df.duplicated().sum()
    print(f"{n_dupes:,} fully duplicated row(s) out of {len(df):,} ({n_dupes / len(df) * 100:.4f}%)")


**What was done:** Defined `profile_dataset()`, a single function that prints shape,
columns, `head()`, `info()`, `describe()`, per-column and total memory usage, a missing-value
report (count + percentage, non-empty columns only), and a full-row duplicate count.

**Why it matters:** Centralizing profiling logic prevents copy-paste drift between the three
dataset sections below (e.g. forgetting `deep=True` on one `memory_usage()` call), and makes
the notebook's diagnostic behavior easy to unit-test later if it's promoted into `src/`.

**ETL decision influenced:** This function is a strong candidate for `src/profiling.py` so the
same audit can be re-run automatically every time new raw data lands (e.g. a new M5-style
weekly refresh), turning this notebook's logic into a repeatable data-quality gate.


## 6. Dataset Profile — `calendar.csv`

### Column reference

| Column | Meaning |
|---|---|
| `date` | Calendar date in `YYYY-MM-DD` format. |
| `wm_yr_wk` | Walmart internal *year-week* identifier (e.g. `11101`). Links to `sell_prices.wm_yr_wk`. |
| `weekday` | Day name (`Monday`, `Tuesday`, …). |
| `wday` | Numeric weekday code (1–7); Walmart's fiscal week starts on Saturday. |
| `month` | Calendar month (1–12). |
| `year` | Calendar year. |
| `d` | The **day key** (`d_1`, `d_2`, …, `d_1913+`) — this is the critical join column, since `sales_train_validation.csv` stores one column per `d_i` instead of one row per day. |
| `event_name_1`, `event_type_1` | Name/type of a primary special event on that date (e.g. `SuperBowl` / `Sporting`), if any. |
| `event_name_2`, `event_type_2` | Name/type of a secondary event, for days with two overlapping events. |
| `snap_CA`, `snap_TX`, `snap_WI` | Binary flag (0/1): whether SNAP (food-assistance) purchases were allowed that day, per state. |

`calendar` is the **time backbone** of the whole dataset: it is the only table with one row
per calendar day, and it is what lets the wide `d_i` sales columns be reshaped into an actual
time series later.


In [14]:
profile_dataset(calendar_df, "calendar")

DATASET: calendar

--- shape ---
1,969 rows x 14 columns

--- columns ---
['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']

--- head(5) ---


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1



--- info() ---
<class 'pandas.DataFrame'>
RangeIndex: 1969 entries, 0 to 1968
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   date          1969 non-null   str  
 1   wm_yr_wk      1969 non-null   int64
 2   weekday       1969 non-null   str  
 3   wday          1969 non-null   int64
 4   month         1969 non-null   int64
 5   year          1969 non-null   int64
 6   d             1969 non-null   str  
 7   event_name_1  162 non-null    str  
 8   event_type_1  162 non-null    str  
 9   event_name_2  5 non-null      str  
 10  event_type_2  5 non-null      str  
 11  snap_CA       1969 non-null   int64
 12  snap_TX       1969 non-null   int64
 13  snap_WI       1969 non-null   int64
dtypes: int64(7), str(7)
memory usage: 688.8 KB

--- describe(include='all').T ---


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
date,1969,1969,2011-01-29,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
wm_yr_wk,"1,969.00",NaN,NaN,NaN,"11,347.09",155.28,"11,101.00","11,219.00","11,337.00","11,502.00","11,621.00"
weekday,1969,7,Saturday,282,NaN,NaN,NaN,NaN,NaN,NaN,NaN
wday,"1,969.00",NaN,NaN,NaN,4.00,2.00,1.00,2.00,4.00,6.00,7.00
month,"1,969.00",NaN,NaN,NaN,6.33,3.42,1.00,3.00,6.00,9.00,12.00
year,"1,969.00",NaN,NaN,NaN,"2,013.29",1.58,"2,011.00","2,012.00","2,013.00","2,015.00","2,016.00"
d,1969,1969,d_1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
event_name_1,162,30,SuperBowl,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
event_type_1,162,4,Religious,55,NaN,NaN,NaN,NaN,NaN,NaN,NaN
event_name_2,5,4,Father's day,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- memory usage (deep) ---
0.67 MB total


,MB
Index,0.00
date,0.11
wm_yr_wk,0.02
weekday,0.11
wday,0.02
month,0.02
year,0.02
d,0.10
event_name_1,0.06
event_type_1,0.06



--- missing values ---


,missing_count,missing_pct
event_type_2,1964,99.75
event_name_2,1964,99.75
event_type_1,1807,91.77
event_name_1,1807,91.77



--- duplicate rows ---
0 fully duplicated row(s) out of 1,969 (0.0000%)


**What was done:** Ran the full profiling routine on `calendar_df`.

**Why it matters:** `calendar` is small but structurally critical — every downstream
time-based feature (day of week, month, SNAP eligibility, holiday proximity) originates here.
Confirming its `d` values are unique and sequential, and understanding which `event_*` /
`snap_*` columns carry nulls, is essential before it can safely be used as a join key.

**ETL decision influenced:** The extent of missing values in `event_name_1/2` and
`event_type_1/2` (expected to be high, since most days have no event) tells the Transform
phase these are **legitimately sparse categorical flags**, not a data-quality defect — they
should be filled with an explicit `"no_event"` sentinel in Transform, *not* dropped.


## 7. Dataset Profile — `sell_prices.csv`

### Column reference

| Column | Meaning |
|---|---|
| `store_id` | Store identifier, formatted `{STATE}_{store_number}` (e.g. `CA_1`). |
| `item_id` | Product identifier, formatted `{DEPT}_{item_number}` (e.g. `HOBBIES_1_001`). |
| `wm_yr_wk` | Year-week identifier — joins back to `calendar.wm_yr_wk` (many `date` rows share one `wm_yr_wk`, since prices are **weekly**, not daily). |
| `sell_price` | Average selling price of the item at that store, for that week, in (unspecified) currency units. |

`sell_prices` is a **fact table at weekly grain** — one row per (store, item, week) — which is
coarser than the daily grain of `sales_train_validation`. That grain mismatch is a key join
consideration for the Transform phase.


In [15]:
profile_dataset(sell_prices_df, "sell_prices")

DATASET: sell_prices

--- shape ---
6,841,121 rows x 4 columns

--- columns ---
['store_id', 'item_id', 'wm_yr_wk', 'sell_price']

--- head(5) ---


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26



--- info() ---
<class 'pandas.DataFrame'>
RangeIndex: 6841121 entries, 0 to 6841120
Data columns (total 4 columns):
 #   Column      Dtype  
---  ------      -----  
 0   store_id    str    
 1   item_id     str    
 2   wm_yr_wk    int64  
 3   sell_price  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 853.1 MB

--- describe(include='all').T ---


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
store_id,6841121,10,TX_2,701214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
item_id,6841121,3049,HOBBIES_1_010,2820,NaN,NaN,NaN,NaN,NaN,NaN,NaN
wm_yr_wk,"6,841,121.00",NaN,NaN,NaN,"11,382.94",148.61,"11,101.00","11,247.00","11,411.00","11,517.00","11,621.00"
sell_price,"6,841,121.00",NaN,NaN,NaN,4.41,3.41,0.01,2.18,3.47,5.84,107.32



--- memory usage (deep) ---
853.13 MB total


,MB
Index,0.00
store_id,345.78
item_id,402.96
wm_yr_wk,52.19
sell_price,52.19



--- missing values ---
No missing values detected in any column.

--- duplicate rows ---
0 fully duplicated row(s) out of 6,841,121 (0.0000%)


**What was done:** Ran the full profiling routine on `sell_prices_df`.

**Why it matters:** This table is typically the **largest** of the three by row count (one row
per store × item × week, across ~5 years), so its memory footprint and dtype choices matter
disproportionately for pipeline performance. Confirming whether `sell_price` has nulls or
negative/zero values here (rather than assuming) determines whether "missing price" is a real
phenomenon in this dataset (e.g. item not yet sold) or a data artifact.

**ETL decision influenced:** Because `sell_prices` is keyed at `(store_id, item_id, wm_yr_wk)`
— **weekly**, not daily — any join into the daily sales/calendar data will necessarily
broadcast one price across ~7 days. That's a modeling assumption the Transform phase must
apply deliberately, not something `pandas` will warn about automatically.


## 8. Dataset Profile — `sales_train_validation.csv`

### Column reference

| Column | Meaning |
|---|---|
| `id` | Unique row identifier, formatted `{item_id}_{store_id}_validation`. One row = one (item, store) time series. |
| `item_id` | Product identifier (same domain as in `sell_prices`). |
| `dept_id` | Department the item belongs to (e.g. `HOBBIES_1`). |
| `cat_id` | Top-level category (`HOBBIES`, `HOUSEHOLD`, `FOODS`). |
| `store_id` | Store identifier (same domain as in `sell_prices`). |
| `state_id` | US state the store is located in (`CA`, `TX`, `WI`) — functionally dependent on `store_id`. |
| `d_1`, `d_2`, …, `d_n` | **Units sold** on that day, one column per day. Column name `d_i` joins to `calendar.d`. This is the dataset's defining "wide" structure. |

This table is the **primary fact table** of the whole project: it is what we are ultimately
trying to forecast. Every other table exists to enrich it (with time attributes or price
attributes).


In [16]:
profile_dataset(sales_df, "sales_train_validation")

DATASET: sales_train_validation

--- shape ---
30,490 rows x 1,919 columns

--- columns ---
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1', 'd_2', 'd_3', 'd_4', 'd_5', 'd_6', 'd_7', 'd_8', 'd_9', 'd_10', 'd_11', 'd_12', 'd_13', 'd_14', 'd_15', 'd_16', 'd_17', 'd_18', 'd_19', 'd_20', 'd_21', 'd_22', 'd_23', 'd_24', 'd_25', 'd_26', 'd_27', 'd_28', 'd_29', 'd_30', 'd_31', 'd_32', 'd_33', 'd_34', 'd_35', 'd_36', 'd_37', 'd_38', 'd_39', 'd_40', 'd_41', 'd_42', 'd_43', 'd_44', 'd_45', 'd_46', 'd_47', 'd_48', 'd_49', 'd_50', 'd_51', 'd_52', 'd_53', 'd_54', 'd_55', 'd_56', 'd_57', 'd_58', 'd_59', 'd_60', 'd_61', 'd_62', 'd_63', 'd_64', 'd_65', 'd_66', 'd_67', 'd_68', 'd_69', 'd_70', 'd_71', 'd_72', 'd_73', 'd_74', 'd_75', 'd_76', 'd_77', 'd_78', 'd_79', 'd_80', 'd_81', 'd_82', 'd_83', 'd_84', 'd_85', 'd_86', 'd_87', 'd_88', 'd_89', 'd_90', 'd_91', 'd_92', 'd_93', 'd_94', 'd_95', 'd_96', 'd_97', 'd_98', 'd_99', 'd_100', 'd_101', 'd_102', 'd_103', 'd_104', 'd_105', 'd_106',

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9,d_10,d_11,d_12,d_13,d_14,d_15,d_16,d_17,d_18,d_19,d_20,d_21,d_22,d_23,d_24,...,d_1884,d_1885,d_1886,d_1887,d_1888,d_1889,d_1890,d_1891,d_1892,d_1893,d_1894,d_1895,d_1896,d_1897,d_1898,d_1899,d_1900,d_1901,d_1902,d_1903,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1,1,1,0,0,0,0,0,1,0,4,2,3,0,1,2,0,0,0,1,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,2,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,6,6,0,0,0,0,3,1,2,1,3,1,0,2,5,4,2,0,3,0,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,4,4,0,1,4,0,1,0,1,0,1,1,2,0,1,1,2,1,1,0,1,1,2,2,2,4



--- info() ---
<class 'pandas.DataFrame'>
RangeIndex: 30490 entries, 0 to 30489
Columns: 1919 entries, id to d_1913
dtypes: int64(1913), str(6)
memory usage: 455.4 MB

--- describe(include='all').T ---


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,30490,30490,HOBBIES_1_001_CA_1_validation,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
item_id,30490,3049,HOBBIES_1_001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dept_id,30490,7,FOODS_3,8230,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cat_id,30490,3,FOODS,14370,NaN,NaN,NaN,NaN,NaN,NaN,NaN
store_id,30490,10,CA_1,3049,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
d_1909,"30,490.00",NaN,NaN,NaN,1.16,2.88,0.00,0.00,0.00,1.00,88.00
d_1910,"30,490.00",NaN,NaN,NaN,1.15,2.95,0.00,0.00,0.00,1.00,77.00
d_1911,"30,490.00",NaN,NaN,NaN,1.33,3.36,0.00,0.00,0.00,1.00,141.00
d_1912,"30,490.00",NaN,NaN,NaN,1.61,4.09,0.00,0.00,0.00,2.00,171.00



--- memory usage (deep) ---
455.38 MB total


,MB
Index,0.00
id,2.26
item_id,1.80
dept_id,1.68
cat_id,1.62
...,...
d_1909,0.23
d_1910,0.23
d_1911,0.23
d_1912,0.23



--- missing values ---
No missing values detected in any column.

--- duplicate rows ---
0 fully duplicated row(s) out of 30,490 (0.0000%)


**What was done:** Ran the full profiling routine on `sales_df`.

**Why it matters:** `sales_train_validation` is stored in **wide format** — thousands of
`d_i` columns instead of a `date`/`quantity` pair — which is efficient for storage but unusable
directly for time-series modeling or for joining against `calendar`/`sell_prices`. Profiling
`describe()` across the `d_i` columns also surfaces early signals of the M5 dataset's known
characteristics: heavy **intermittency** (many zeros) and **right-skew** (occasional large
spikes), both of which matter for model choice later.

**ETL decision influenced:** Confirms this table needs to be **melted from wide to long**
(one row per `id` × `d`) before it can be joined to `calendar` on `d` and, transitively, to
`sell_prices` on `(store_id, item_id, wm_yr_wk)`. That reshape is explicitly deferred to the
Transform phase — this notebook only documents *why* it will be necessary.


## 9. Inspecting Every Identifier Column

Identifier columns (`id`, `item_id`, `dept_id`, `cat_id`, `store_id`, `state_id`,
`wm_yr_wk`, `d`, `date`) are the columns every join and every key decision depends on. For
each one we check: cardinality (`nunique`), whether it's unique per row, its value format, and
a small sample of values — rather than assuming their structure from documentation alone.


In [17]:
def inspect_identifier(df: pd.DataFrame, col: str, dataset_name: str) -> dict:
    series = df[col]
    n_unique = series.nunique(dropna=False)
    n_rows = len(series)
    return {
        "dataset": dataset_name,
        "column": col,
        "dtype": str(series.dtype),
        "n_unique": n_unique,
        "n_rows": n_rows,
        "is_unique_per_row": n_unique == n_rows,
        "n_nulls": int(series.isna().sum()),
        "sample_values": series.dropna().unique()[:3].tolist(),
    }


identifier_columns = [
    ("calendar", calendar_df, ["date", "wm_yr_wk", "d"]),
    ("sell_prices", sell_prices_df, ["store_id", "item_id", "wm_yr_wk"]),
    ("sales_train_validation", sales_df, ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]),
]

id_reports = []
for dataset_name, df, cols in identifier_columns:
    for col in cols:
        id_reports.append(inspect_identifier(df, col, dataset_name))

identifier_report = pd.DataFrame(id_reports)
display(identifier_report)


,dataset,column,dtype,n_unique,n_rows,is_unique_per_row,n_nulls,sample_values
0,calendar,date,str,1969,1969,True,0,"[2011-01-29, 2011-01-30, 2011-01-31]"
1,calendar,wm_yr_wk,int64,282,1969,False,0,"[11101, 11102, 11103]"
2,calendar,d,str,1969,1969,True,0,"[d_1, d_2, d_3]"
3,sell_prices,store_id,str,10,6841121,False,0,"[CA_1, CA_2, CA_3]"
4,sell_prices,item_id,str,3049,6841121,False,0,"[HOBBIES_1_001, HOBBIES_1_002, HOBBIES_1_003]"
5,sell_prices,wm_yr_wk,int64,282,6841121,False,0,"[11325, 11326, 11327]"
6,sales_train_validation,id,str,30490,30490,True,0,"[HOBBIES_1_001_CA_1_validation, HOBBIES_1_002_..."
7,sales_train_validation,item_id,str,3049,30490,False,0,"[HOBBIES_1_001, HOBBIES_1_002, HOBBIES_1_003]"
8,sales_train_validation,dept_id,str,7,30490,False,0,"[HOBBIES_1, HOBBIES_2, HOUSEHOLD_1]"
9,sales_train_validation,cat_id,str,3,30490,False,0,"[HOBBIES, HOUSEHOLD, FOODS]"


**What was done:** For every identifier-style column across all three tables, computed
cardinality, row count, whether the column is unique per row, null count, and a small value
sample — laid out as a single comparison table.

**Why it matters:** This is the evidence base for every claim we're about to make about
primary keys, foreign keys, and grain. A key claim like *"`id` uniquely identifies a row in
`sales_train_validation`"* should be **verified** (`is_unique_per_row == True`), not assumed
from the M5 competition documentation, since real-world extracts can diverge from spec
(duplicate rows, re-exports, truncated files, etc.).

**ETL decision influenced:** Directly feeds section 11 (Primary & Foreign Keys) below — any
column here that unexpectedly shows `is_unique_per_row == False` where uniqueness was assumed
would block a Transform-phase merge until investigated, since it would produce row
duplication (a "fan-out") on join.


## 10. Discovering Relationships Between Datasets

We now check how the three tables actually connect, using set-overlap and dependency checks
**on the real data** rather than the documented schema alone.

- `calendar.d` ↔ the `d_i` column names of `sales_train_validation` (day key).
- `calendar.wm_yr_wk` ↔ `sell_prices.wm_yr_wk` (week key).
- `sales_train_validation.item_id` / `store_id` ↔ `sell_prices.item_id` / `store_id` (product & store keys).
- Functional dependency: does `store_id` always imply the same `state_id`?


In [18]:
# --- calendar.d vs the d_i columns present in sales_train_validation ---
sales_day_cols = [c for c in sales_df.columns if c.startswith("d_")]
calendar_days = set(calendar_df["d"])
sales_days = set(sales_day_cols)

print("Day-key relationship: calendar.d <-> sales_train_validation d_i columns")
print(f"  distinct days in calendar          : {len(calendar_days):,}")
print(f"  d_i columns in sales_train_validation: {len(sales_days):,}")
print(f"  days in calendar but NOT in sales    : {len(calendar_days - sales_days):,}")
print(f"  d_i columns in sales but NOT in calendar: {len(sales_days - calendar_days):,}")

# --- calendar.wm_yr_wk vs sell_prices.wm_yr_wk ---
cal_weeks = set(calendar_df["wm_yr_wk"])
price_weeks = set(sell_prices_df["wm_yr_wk"])
print("\nWeek-key relationship: calendar.wm_yr_wk <-> sell_prices.wm_yr_wk")
print(f"  distinct weeks in calendar    : {len(cal_weeks):,}")
print(f"  distinct weeks in sell_prices : {len(price_weeks):,}")
print(f"  weeks in sell_prices but NOT in calendar: {len(price_weeks - cal_weeks):,}")

# --- item_id / store_id overlap between sales and sell_prices ---
print("\nProduct/store-key relationship: sales_train_validation <-> sell_prices")
sales_items, price_items = set(sales_df["item_id"]), set(sell_prices_df["item_id"])
sales_stores, price_stores = set(sales_df["store_id"]), set(sell_prices_df["store_id"])
print(f"  item_id overlap : {len(sales_items & price_items):,} shared "
      f"({len(sales_items - price_items):,} sales-only, {len(price_items - sales_items):,} prices-only)")
print(f"  store_id overlap: {len(sales_stores & price_stores):,} shared "
      f"({len(sales_stores - price_stores):,} sales-only, {len(price_stores - sales_stores):,} prices-only)")

# --- functional dependency: store_id -> state_id ---
fd_check = sales_df.groupby("store_id")["state_id"].nunique()
violations = fd_check[fd_check > 1]
print("\nFunctional dependency check: store_id -> state_id")
if violations.empty:
    print("  Holds for all stores: every store_id maps to exactly one state_id.")
else:
    print(f"  VIOLATED for {len(violations)} store_id value(s):")
    display(violations)


Day-key relationship: calendar.d <-> sales_train_validation d_i columns
  distinct days in calendar          : 1,969
  d_i columns in sales_train_validation: 1,913
  days in calendar but NOT in sales    : 56
  d_i columns in sales but NOT in calendar: 0

Week-key relationship: calendar.wm_yr_wk <-> sell_prices.wm_yr_wk
  distinct weeks in calendar    : 282
  distinct weeks in sell_prices : 282
  weeks in sell_prices but NOT in calendar: 0

Product/store-key relationship: sales_train_validation <-> sell_prices
  item_id overlap : 3,049 shared (0 sales-only, 0 prices-only)
  store_id overlap: 10 shared (0 sales-only, 0 prices-only)

Functional dependency check: store_id -> state_id
  Holds for all stores: every store_id maps to exactly one state_id.


**What was done:** Measured the *actual* set overlap between the day key
(`calendar.d` vs. the sales table's `d_i` column names), the week key
(`calendar.wm_yr_wk` vs. `sell_prices.wm_yr_wk`), and the product/store keys
(`item_id`/`store_id` shared between `sales_train_validation` and `sell_prices`). Also tested
the functional dependency `store_id → state_id`.

**Why it matters:** A join is only safe when the key domains actually line up. For example, if
`sell_prices` contained `wm_yr_wk` values absent from `calendar`, a join would silently drop
rows (or need an explicit `how="left"` decision) — something we want to *know*, not discover
after modeling. The `store_id → state_id` check confirms `state_id` is redundant information
that could be derived from `store_id` alone (a normalization/dimension-table signal).

**ETL decision influenced:** Confirms the join path the Transform phase should implement:
melt `sales_train_validation` → merge onto `calendar` via `d` → merge the result onto
`sell_prices` via `(store_id, item_id, wm_yr_wk)`. Any overlap gaps surfaced above dictate
whether those merges should be `inner` (drop unmatched) or `left` (keep unmatched, expect
nulls) joins.


## 11. Primary Keys and Foreign Keys

Based on the identifier inspection (§9) and relationship discovery (§10) above, the key
structure is:

| Table | Primary Key (grain) | Foreign Keys |
|---|---|---|
| `calendar` | `d` (equivalently `date`, 1:1) — one row per **day** | *none* (calendar is the root time dimension) |
| `sell_prices` | composite: `(store_id, item_id, wm_yr_wk)` — one row per **store × item × week** | `wm_yr_wk` → `calendar.wm_yr_wk` |
| `sales_train_validation` | `id` — one row per **item × store** series (before melting); after melting to long format it becomes composite `(id, d)` | `item_id`, `store_id` (composite, → `sell_prices`); `d` → `calendar.d` (only after melting) |

Notes:
- `sales_train_validation.id` is *itself* a concatenation of `item_id` + `store_id` + the
  literal suffix `"validation"` — i.e. it is a **derived/surrogate key**, not an independently
  assigned one. `item_id` + `store_id` together already uniquely identify a series.
- There is no single column in `sell_prices` that is unique per row; uniqueness only holds for
  the **composite** `(store_id, item_id, wm_yr_wk)`, which was verified implicitly by row-count
  vs. cardinality comparisons in §9/§10 and confirmed with an explicit duplicate check below.


In [19]:
composite_key_cols = ["store_id", "item_id", "wm_yr_wk"]
n_rows = len(sell_prices_df)
n_unique_composite = sell_prices_df[composite_key_cols].drop_duplicates().shape[0]

print("Composite key check for sell_prices: (store_id, item_id, wm_yr_wk)")
print(f"  total rows            : {n_rows:,}")
print(f"  unique combinations   : {n_unique_composite:,}")
print(f"  composite key is valid: {n_rows == n_unique_composite}")

print("\nPrimary key check for sales_train_validation: id")
print(f"  is 'id' unique per row: {sales_df['id'].is_unique}")


Composite key check for sell_prices: (store_id, item_id, wm_yr_wk)
  total rows            : 6,841,121
  unique combinations   : 6,841,121
  composite key is valid: True

Primary key check for sales_train_validation: id
  is 'id' unique per row: True


**What was done:** Explicitly verified that `(store_id, item_id, wm_yr_wk)` has zero
duplicate combinations in `sell_prices` (i.e. it is a valid composite primary key), and that
`id` is unique per row in `sales_train_validation`.

**Why it matters:** "Primary key" claims are only trustworthy once verified programmatically —
this table doesn't come with database-enforced constraints (it's a CSV), so uniqueness must be
checked rather than assumed from the competition documentation.

**ETL decision influenced:** This key map is what the Transform phase's `merge()` calls should
be built against verbatim: `on="d"` for calendar↔sales, `on=["store_id","item_id","wm_yr_wk"]`
for the result↔sell_prices. It also tells the Load phase which columns need a uniqueness
constraint (or composite index) if the target is a relational warehouse table.


## 12. Dimension Tables vs. Fact Tables

Viewed through a star-schema lens:

| Table | Role | Reasoning |
|---|---|---|
| `calendar` | **Dimension** (time dimension) | One row per day; descriptive attributes (weekday, month, events, SNAP flags) about *when*, not a measured quantity. Low row count relative to the fact table. |
| `sell_prices` | **Hybrid** — weekly price *fact*, but also functions as a **product/store price-lookup dimension** for the daily grain | Contains a measure (`sell_price`) but at a coarser (weekly) grain than the core fact table, so relative to `sales_train_validation` it behaves like a dimension to be looked up. |
| `sales_train_validation` (melted) | **Fact table** | Contains the actual measured quantity we care about — units sold — at the finest available grain (item × store × day), with foreign keys out to every dimension. |

An implied **product dimension** (`item_id`, `dept_id`, `cat_id`) and **store/location
dimension** (`store_id`, `state_id`) are currently embedded *inside* the fact table rather than
normalized into their own tables — worth flagging for the Transform/Load phases if a proper
star schema is desired downstream (e.g. for a BI/warehouse layer).


In [20]:
# Evidence: product & location attributes are repeated (denormalized) across many rows
# of the fact table, which is exactly what a proper dimension table would remove.
product_dim_preview = sales_df[["item_id", "dept_id", "cat_id"]].drop_duplicates()
location_dim_preview = sales_df[["store_id", "state_id"]].drop_duplicates()

print(f"Fact table rows                 : {len(sales_df):,}")
print(f"Distinct (item_id, dept_id, cat_id) combos -> implied product dimension size: {len(product_dim_preview):,}")
print(f"Distinct (store_id, state_id) combos       -> implied store dimension size : {len(location_dim_preview):,}")

display(product_dim_preview.head())
display(location_dim_preview.head())


Fact table rows                 : 30,490
Distinct (item_id, dept_id, cat_id) combos -> implied product dimension size: 3,049
Distinct (store_id, state_id) combos       -> implied store dimension size : 10


,item_id,dept_id,cat_id
0,HOBBIES_1_001,HOBBIES_1,HOBBIES
1,HOBBIES_1_002,HOBBIES_1,HOBBIES
2,HOBBIES_1_003,HOBBIES_1,HOBBIES
3,HOBBIES_1_004,HOBBIES_1,HOBBIES
4,HOBBIES_1_005,HOBBIES_1,HOBBIES


,store_id,state_id
0,CA_1,CA
3049,CA_2,CA
6098,CA_3,CA
9147,CA_4,CA
12196,TX_1,TX


**What was done:** Quantified how much repetition exists in the product-related
(`item_id`, `dept_id`, `cat_id`) and location-related (`store_id`, `state_id`) attributes
inside the fact table, by counting their distinct combinations relative to total row count.

**Why it matters:** The large gap between total fact rows and the small number of distinct
product/location combinations is direct evidence of denormalization — those attributes are
dimension-like even though they currently live inside the fact table.

**ETL decision influenced:** If a Load-phase target is a dimensional warehouse (star schema),
this is the evidence to extract `dim_product` and `dim_store` tables during Transform (via
`drop_duplicates()` on the columns shown above) rather than repeating those attributes on every
fact row.


## 13. Dataset Size and Memory Requirements

We already captured per-dataset memory usage inside `profile_dataset()`. Here we consolidate
those numbers into a single summary and — importantly — **project forward** to the memory that
the *melted* (long-format) version of `sales_train_validation` will require, since that reshape
is known to be coming in the Transform phase (§8) and is by far the largest driver of memory
pressure in this pipeline.


In [21]:
summary_rows = []
for name, df in raw_datasets.items():
    summary_rows.append({
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1],
        "memory_MB_shallow": df.memory_usage(deep=False).sum() / (1024 ** 2),
        "memory_MB_deep": df.memory_usage(deep=True).sum() / (1024 ** 2),
    })

memory_summary = pd.DataFrame(summary_rows)
memory_summary["total_MB_deep"] = memory_summary["memory_MB_deep"]
display(memory_summary)

print(f"\nCombined raw in-memory footprint (deep): {memory_summary['memory_MB_deep'].sum():,.2f} MB")

# Forward-looking projection: melting sales_train_validation from wide to long
# multiplies row count by the number of day columns and collapses to ~3 columns
# (id, d, sales) plus the untouched categorical id columns repeated per row.
n_day_cols = len([c for c in sales_df.columns if c.startswith("d_")])
projected_long_rows = len(sales_df) * n_day_cols
print(f"\nProjected row count AFTER melting sales_train_validation to long format: {projected_long_rows:,} rows")
print("(Actual melt is deferred to the Transform phase; this is a sizing estimate only.)")


,dataset,rows,columns,memory_MB_shallow,memory_MB_deep,total_MB_deep
0,calendar,1969,14,0.21,0.67,0.67
1,sell_prices,6841121,4,208.77,853.13,853.13
2,sales_train_validation,30490,1919,446.40,455.38,455.38



Combined raw in-memory footprint (deep): 1,309.19 MB

Projected row count AFTER melting sales_train_validation to long format: 58,327,370 rows
(Actual melt is deferred to the Transform phase; this is a sizing estimate only.)


**What was done:** Consolidated shallow vs. deep memory usage across all three raw
tables into one summary table, and computed a forward-looking row-count projection for what
`sales_train_validation` will grow to once melted from wide to long format (rows ×
number of `d_i` columns).

**Why it matters:** Deep memory usage (which accounts for actual string/object payload size,
not just pointer size) is the number that matters for capacity planning — shallow usage
systematically understates memory for any `object`-dtype column. The melt projection matters
because wide-to-long reshapes are exactly the kind of operation that silently balloons memory
if not planned for (going from tens of thousands of rows to tens of millions).

**ETL decision influenced:** If the projected melted row count is large relative to available
RAM, this is the evidence to decide — **before** the Transform phase is built — whether it
needs chunked processing (e.g. `pd.read_csv(..., chunksize=...)`, per-store processing loops,
or a switch to Dask/Polars) rather than a single `pd.melt()` call on the full DataFrame.


## 14. Potential Data Quality Issues

A structured sweep for the data-quality categories most relevant to a forecasting pipeline:
missing values (recap), duplicate rows, referential gaps, and value-range sanity checks
(negative/zero prices, negative sales, unexpected data types).


In [22]:
issues = []

# 1) Missing values recap (counts already computed inside profile_dataset per table)
for name, df in raw_datasets.items():
    n_missing_cols = (df.isna().sum() > 0).sum()
    issues.append({
        "check": "missing values",
        "dataset": name,
        "finding": f"{n_missing_cols} column(s) contain at least one null",
        "severity": "info" if n_missing_cols == 0 else "review",
    })

# 2) Duplicate rows recap
for name, df in raw_datasets.items():
    n_dupes = df.duplicated().sum()
    issues.append({
        "check": "duplicate rows",
        "dataset": name,
        "finding": f"{n_dupes} fully duplicated row(s)",
        "severity": "info" if n_dupes == 0 else "review",
    })

# 3) sell_price sanity: non-positive prices
non_positive_prices = (sell_prices_df["sell_price"] <= 0).sum()
issues.append({
    "check": "value range (sell_price <= 0)",
    "dataset": "sell_prices",
    "finding": f"{non_positive_prices} row(s) with sell_price <= 0",
    "severity": "info" if non_positive_prices == 0 else "review",
})

# 4) sales sanity: negative units sold across all d_i columns
day_cols = [c for c in sales_df.columns if c.startswith("d_")]
negative_sales_cells = (sales_df[day_cols] < 0).to_numpy().sum()
issues.append({
    "check": "value range (units sold < 0)",
    "dataset": "sales_train_validation",
    "finding": f"{negative_sales_cells} cell(s) with negative units sold across all day columns",
    "severity": "info" if negative_sales_cells == 0 else "review",
})

# 5) Referential gap: weeks present in sell_prices but absent from calendar
weeks_gap = len(set(sell_prices_df["wm_yr_wk"]) - set(calendar_df["wm_yr_wk"]))
issues.append({
    "check": "referential integrity (sell_prices.wm_yr_wk -> calendar.wm_yr_wk)",
    "dataset": "sell_prices / calendar",
    "finding": f"{weeks_gap} week value(s) in sell_prices not found in calendar",
    "severity": "info" if weeks_gap == 0 else "review",
})

# 6) Sparsity / intermittency signal in sales (relevant for model choice, not a "defect")
zero_fraction = (sales_df[day_cols] == 0).to_numpy().mean()
issues.append({
    "check": "sparsity (fraction of zero-sales cells)",
    "dataset": "sales_train_validation",
    "finding": f"{zero_fraction:.2%} of all (item, store, day) cells equal zero",
    "severity": "info",
})

# 7) High-cardinality / mostly-null event columns (expected pattern, not a defect)
for col in ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]:
    pct_null = calendar_df[col].isna().mean()
    issues.append({
        "check": "sparse categorical column",
        "dataset": "calendar",
        "finding": f"'{col}' is {pct_null:.1%} null",
        "severity": "info",
    })

quality_report = pd.DataFrame(issues)
display(quality_report)


,check,dataset,finding,severity
0,missing values,calendar,4 column(s) contain at least one null,review
1,missing values,sell_prices,0 column(s) contain at least one null,info
2,missing values,sales_train_validation,0 column(s) contain at least one null,info
3,duplicate rows,calendar,0 fully duplicated row(s),info
4,duplicate rows,sell_prices,0 fully duplicated row(s),info
5,duplicate rows,sales_train_validation,0 fully duplicated row(s),info
6,value range (sell_price <= 0),sell_prices,0 row(s) with sell_price <= 0,info
7,value range (units sold < 0),sales_train_validation,0 cell(s) with negative units sold across all ...,info
8,referential integrity (sell_prices.wm_yr_wk ->...,sell_prices / calendar,0 week value(s) in sell_prices not found in ca...,info
9,sparsity (fraction of zero-sales cells),sales_train_validation,"68.20% of all (item, store, day) cells equal zero",info


**What was done:** Ran a checklist-style sweep covering: missing values, duplicate
rows, non-positive prices, negative sales quantities, a referential-integrity check between
`sell_prices.wm_yr_wk` and `calendar.wm_yr_wk`, the overall zero-sales sparsity rate, and the
null rate of the four event columns — tagging each finding with a severity (`info` vs.
`review`).

**Why it matters:** M5 is a well-known benchmark dataset, but "well-known" doesn't mean
"assume it's clean" — a production pipeline should verify these properties against the actual
file on disk every time, since raw exports, re-downloads, or partial files can silently differ
from expectations. Distinguishing `info` (expected pattern, e.g. sparse events) from `review`
(needs a human decision, e.g. non-positive prices) keeps the report actionable rather than
alarming.

**ETL decision influenced:** Any `review`-tagged row here becomes an explicit, documented
decision the Transform phase must make (e.g. "drop rows with `sell_price <= 0`" or "treat as
promotional/free items and keep them") — decided with evidence, not silently overwritten by a
generic `.dropna()` / `.fillna(0)` call.


## 15. Extract Phase Decisions — Summary

This section consolidates everything discovered above into the concrete facts and decisions
the **Transform phase** will build on.

### 15.1 Confirmed facts about the raw data

- **Three raw tables** were located and loaded successfully via a portable, marker-based
  `pathlib` project-root resolver (no absolute paths).
- **`calendar`** is a time dimension at **daily grain**, keyed by `d` (⟷ `date`, 1:1).
- **`sell_prices`** is keyed by the **composite** `(store_id, item_id, wm_yr_wk)` — weekly
  grain, one level coarser than daily.
- **`sales_train_validation`** is the **fact table**, currently stored **wide** (one column per
  day, `d_1..d_n`), keyed by `id` (derivable from `item_id` + `store_id`).
- `store_id → state_id` is a **functional dependency** (verified) — `state_id` is redundant
  information relative to `store_id`.
- Product attributes (`item_id`, `dept_id`, `cat_id`) and location attributes (`store_id`,
  `state_id`) are **denormalized** into the fact table — candidates for `dim_product` /
  `dim_store` extraction.
- Event columns (`event_name_1/2`, `event_type_1/2`) are **legitimately sparse**, not broken —
  most days have no event.
- Sales data is expected to be **highly intermittent** (a large share of zero-sales
  item/store/day cells) — see the sparsity metric in §14.

### 15.2 Key structure to carry into Transform

| Join | Keys | Type |
|---|---|---|
| `sales_train_validation` (melted) → `calendar` | `d` | many-to-one |
| result → `sell_prices` | `(store_id, item_id, wm_yr_wk)` | many-to-one |

### 15.3 Decisions this phase hands off to Transform

1. **Reshape** `sales_train_validation` from wide (`d_1..d_n` columns) to long
   (`id`, `d`, `sales` rows) — required before any date-based join or time-series operation.
2. **Join order**: melt sales → merge with `calendar` on `d` → merge with `sell_prices` on
   `(store_id, item_id, wm_yr_wk)`.
3. **Dtype optimization** (e.g. downcasting sales counts to `int16`/`int32`, prices to
   `float32`, high-repetition string columns to `category`) should happen in Transform, sized
   against the memory projection in §13 — not guessed at up front.
4. **Missing-value strategy for events**: fill `event_name_*` / `event_type_*` with an explicit
   sentinel category (e.g. `"none"`), do **not** drop rows or columns.
5. **Any `review`-severity findings from §14** (non-positive prices, negative sales, referential
   gaps, if present in the real dataset) must be resolved with an explicit, documented rule
   before Transform proceeds — this notebook intentionally leaves them **unresolved** and
   visible.
6. **Dimensional modeling** (optional, if a warehouse/BI Load target is planned): extract
   `dim_product` and `dim_store` from the fact table using the `drop_duplicates()` patterns
   demonstrated in §12.
7. **Scalability**: if the melted row-count projection from §13 is large relative to available
   memory, Transform should process in chunks (e.g. per `store_id`) rather than materializing
   one full long-format DataFrame in memory.

### 15.4 Explicit non-goals of this notebook (by design)

- No data was cleaned, imputed, cast, or dropped.
- No files were written to `data/processed/` or anywhere else.
- No merges were materialized — only their *feasibility and correctness* were verified.
- No modeling or feature engineering occurred.

---
**Next notebook in the pipeline:** `notebooks/02_transform.ipynb`
